In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, the fictional capital of the moon, including its weather and demographic details.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The current weather in Lunapolis is described as clear skies, with a high temperature of 120C and a low of -100C.\n- There are 100,000 cheese miners residing in Lunapolis.\n- The cheese miners' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='14cc50ff-c037-4efe-9439-102b8da621a8'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='1bdbac76-6ede-430d-9b91-d24d10c7f9e6'),
              AIMessage(conte

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, the fictional capital of the moon, including its weather and demographic details.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The current weather in Lunapolis is described as clear skies, with a high temperature of 120C and a low of -100C.
- There are 100,000 cheese miners residing in Lunapolis.
- The cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='99583ee4-6185-473b-b3f4-e4810326e9a6'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='3842b103-e570-4ed7-a948-45dba7ebadcb', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='285cf81e-7131-4e7c-8959-0cf414cccaaa'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='bf11196a-a47e-4fa3-bfd3-75f6ed31b09b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='e5b17992-1e89-47c0-9333-a52998e77a66'),
              AIMessage(content='I can’t read an internal temperature if the device isn’t power

In [8]:
print(response["messages"][-1].content)

I can’t read an internal temperature if the device isn’t powered on. Temperature readings require the system to be running (or a special external tool). If it’s not turning on, you won’t be able to check temps with software.

What you can do next to diagnose power-on issues:
- Try a different power outlet and a known-good charger/cable (and check the charger for any damage).
- Look for any lights or sounds when you plug it in or press the power button (beeps, fans starting, LED codes) and note the pattern.
- If it’s a laptop with a removable battery: unplug, remove the battery, hold the power button for 15–20 seconds, reconnect power (without the battery) and try turning it on.
- For desktops: ensure the power switch on the back is ON, unplug and reseat internal cables, RAM, and any graphics card; try booting with minimal components.
- If there are no signs of life after trying these, there may be a hardware failure (power supply, motherboard, battery, etc.) and you might need service.